### ⚙️ Basic Setup

In [0]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import seaborn as sns

# style like R ggplot
plt.style.use("ggplot")

In [0]:
# base volume path
BASE_DIR = "/Volumes/workspace/default/home-credit-default-risk"

### 🏢 Bureau
The `bureau` table contains data on a customer's past borrowing history with other financial institutions (banks, credit unions, or other lenders) outside of Home Credit, as reported to the central credit bureau. Each row represents a single loan or credit line from another bank. It includes details such as whether that external account is currently active or closed, the original credit amount, how much debt remains, and if the borrower ever went overdue on payments with those other lenders.

In [0]:
# reading data
bureau = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_DIR}/bureau.csv")
)

In [0]:
# shape of data
print(f"({bureau.count()}, {len(bureau.columns)})")

In [0]:
bureau.printSchema()

#### ✏️ Table Description

##### 1. SK_ID_CURR
* **Description:** Unique identifier for the current Home Credit loan application.
* **Interpretation:** Foreign key linking back to the primary applicant in `application_train`. A single `SK_ID_CURR` can have multiple external loans in this table (1:N relationship).

##### 2. SK_ID_BUREAU
* **Description:** Unique identifier for each specific loan reported by the Credit Bureau.
* **Interpretation:** Primary key of this table and foreign key used to join with the `bureau_balance` child table (1:N relationship).

##### 3. CREDIT_ACTIVE
* **Description:** Current status of the reported external loan.
* **Interpretation:** Categorical indicator showing if the credit line is ongoing or completed (`Active`, `Closed`, `Sold`, or `Bad debt`). Active loans carry current repayment obligations.

##### 4. CREDIT_CURRENCY
* **Description:** Currency in which the external credit was issued.
* **Interpretation:** Categorical variable (e.g., `currency 1`, `currency 2`) indicating local or foreign currency exposure.

##### 5. DAYS_CREDIT
* **Description:** Time elapsed since the external loan was applied for, relative to the current application date.
* **Interpretation:** Time location in negative days. For example, `-300` means the client opened this credit account 300 days ago. Indicates credit history length and recency of borrowing.

##### 6. DAYS_CREDIT_ENDDATE
* **Description:** Expected remaining or passed duration of the external loan at the time of application.
* **Interpretation:** Negative values mean the loan contract has already expired; positive values mean the loan is scheduled to end in the future.

##### 7. DAYS_ENDDATE_FACT
* **Description:** Actual date when the external loan ended, for closed loans.
* **Interpretation:** Negative days relative to the current application. Null for loans that are still open (`CREDIT_ACTIVE == 'Active'`).

##### 8. DAYS_CREDIT_UPDATE
* **Description:** How many days before the current application the Credit Bureau last updated information about this loan.
* **Interpretation:** Recency indicator of bureau reporting. A value of `-5` means the bureau updated this record 5 days ago.

##### 9. CREDIT_DAY_OVERDUE
* **Description:** Current number of days the loan is past due at the time of data collection.
* **Interpretation:** Direct measure of active delinquency duration. A value of `0` means current on payments; values $> 0$ signal active delinquency.

##### 10. AMT_CREDIT_MAX_OVERDUE
* **Description:** Maximum dollar amount that was ever overdue on this specific loan account.
* **Interpretation:** Historical worst-case delinquency amount. Helps assess the severity of past defaults.

##### 11. AMT_CREDIT_SUM_OVERDUE
* **Description:** Total dollar amount currently overdue on this loan.
* **Interpretation:** Immediate financial distress signal. High active overdue amounts indicate high risk for new credit approval.

##### 12. CNT_CREDIT_PROLONG
* **Description:** Number of times the loan duration/expiry date was extended or prolonged.
* **Interpretation:** High prolongation counts suggest the borrower struggled to repay on the original terms and required contract extensions.

##### 13. AMT_CREDIT_SUM
* **Description:** Total credit amount extended or approved for this loan account.
* **Interpretation:** Reflects historical borrowing capacity and overall exposure across external institutions.

##### 14. AMT_CREDIT_SUM_DEBT
* **Description:** Current outstanding debt remaining on this loan account.
* **Interpretation:** Total unpaid principal owed today. High debt levels relative to income indicate potential over-leveraging.

##### 15. AMT_CREDIT_SUM_LIMIT
* **Description:** Current available credit limit on revolving credit accounts (e.g., credit cards).
* **Interpretation:** Represents unused, instantly accessible purchasing power.

##### 16. AMT_ANNUITY
* **Description:** Monthly payment installment amount for this external loan.
* **Interpretation:** Regular monthly debt obligation. Used to calculate overall monthly debt service coverage when combined with new application requests.

##### 17. CREDIT_TYPE
* **Description:** Category of credit extended by the bureau/external bank.
* **Interpretation:** Indicates loan purpose and risk profile (e.g., `Consumer credit`, `Credit card`, `Car loan`, `Mortgage`, `Microloan`). Microloans and credit cards generally carry higher default risks than mortgages or car loans.

In [0]:
bureau.show(5)

---

#### Nature of SK_ID_CURR and SK_ID_BUREAU
`SK_ID_CURR` can have duplicates. Since one applicant can take out multiple loans at outside banks over time, the same `SK_ID_CURR` appears once for every external loan they have ever held.

`SK_ID_BUREAU` cannot have duplicates. It is the Primary Key of the bureau table. Every single external loan account gets its own unique ID, so each `SK_ID_BUREAU` value appears exactly once in this table.

#### Central Tendency of Applicant Engagement

In [0]:
bureau.groupBy("SK_ID_CURR") \
      .count() \
      .agg(
          sf.avg("count").alias("mean_count"),
          sf.expr("percentile_approx(count, 0.5)").alias("median_count") # using approx percentile for performance
      ) \
      .show()

Because the mean exceeds the median, standard distribution properties indicate that the record counts per `SK_ID_CURR` in the `bureau` table are right-skewed. This reflects a concentration of applicants with fewer external loans, alongside a long right tail of high-frequency borrowers.

#### 👤 Profile of the Most Active Applicant in Bureau History

In [0]:
# aggregate top 10 customers and collect to Pandas
top_10_customers_df = (
    bureau
    .groupBy("SK_ID_CURR")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
    .toPandas()
)

# Convert SK_ID_CURR to string for proper categorical rendering
top_10_customers_df["SK_ID_CURR"] = top_10_customers_df["SK_ID_CURR"].astype(str)

fig, ax = plt.subplots(figsize=(12, 6))

sns.barplot(
    data=top_10_customers_df,
    x="SK_ID_CURR",
    y="count",
    hue="SK_ID_CURR",
    legend=False,
    palette="viridis",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Top 10 Applicants by Total External Bureau Loans",
    fontsize=14,
    fontweight="bold",
    pad=15
)
ax.set_xlabel("Applicant ID (SK_ID_CURR)", fontsize=12)
ax.set_ylabel("Total Loan Count", fontsize=12)
ax.tick_params(axis="x", rotation=45)

ax.grid(axis="y", linestyle="--", alpha=0.7)

for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3)

fig.tight_layout()
plt.show()

#### Applicant Engagement Across Bureau Records

The presence of multiple records for the same applicant (`SK_ID_CURR`) in the `bureau` table demonstrates that many clients actively maintain multiple financial relationships across outside institutions. Rather than being one-time borrowers, these applicants are continuously engaged with the credit market through various ongoing or past loans, credit cards, and financing arrangements.

---

#### Interpretation of `DAYS_CREDIT`

`DAYS_CREDIT` captures the exact age of an external credit account by recording how many days prior to the current Home Credit application it was opened. Represented as a negative integer relative to application day (`Day 0`), a value of `-365` indicates a loan opened exactly one year ago, while `-31` represents a loan taken out just a month prior. 

From a credit risk perspective, this timeline variable serves two crucial modeling functions:
* **Credit History Length:** The minimum value (e.g., `-2000`) establishes an applicant's financial track record, where a longer history of managed accounts signals financial stability and lower risk.
* **Borrowing Velocity:** A concentration of recent accounts (e.g., multiple loans with `DAYS_CREDIT` between `-1` and `-30`) alerts the model to sudden credit-seeking behavior, often a strong indicator of acute liquidity distress and elevated default risk.

In [0]:
# histplot
bureau.select("DAYS_CREDIT").plot.hist(title="Histogram Plot of Days Credit")

In [0]:
# box plot
bureau.select("DAYS_CREDIT").plot.box()

---

#### Distribution of Active Credits

In [0]:
credit_active_counts_df = (
    bureau
    .groupBy("CREDIT_ACTIVE")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=credit_active_counts_df,
    x="CREDIT_ACTIVE",
    y="count",
    hue="CREDIT_ACTIVE",
    legend=False,
    palette="Blues",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Distribution of Credit Active Status",
    fontsize=14,
    fontweight="bold",
    pad=15
)
ax.set_xlabel("Credit Active Status", fontsize=12)
ax.set_ylabel("Total Record Count (Log Scale)", fontsize=12)

ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3)

fig.tight_layout()
plt.show()

In [0]:
credit_active_counts_df

#### 💲What is Sold Accounts and How it works??
The 6,527 "Sold" accounts represent cases where external banks gave up on collecting debt directly and sold the delinquent loans to third-party debt collection agencies.

`For example`:  
1. When you owe a bank $5,000 and stop making payments for 6 to 12 months:

2. The Bank Gives Up: The bank realizes spending time and money calling you isn't working. They write off the $5,000 as a loss on their accounting books.

3. The Sale: A specialized Debt Collection Agency steps in and offers to buy that debt from the bank for cheap—say, $500 (10 cents on the dollar).

4. The Transfer: The bank takes the $500, officially transfers ownership of your loan contract to the agency, and marks your bureau status as "Sold".

5. The New Owner: The bank is completely out of the picture. The collection agency now legally owns your $5,000 debt and will try to collect it from you to make a profit.

---

#### Distribution of Credit Currency

In [0]:
credit_currency_counts_df = (
    bureau
    .groupBy("CREDIT_CURRENCY")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=credit_currency_counts_df,
    x="CREDIT_CURRENCY",
    y="count",
    hue="CREDIT_CURRENCY",
    legend=False,
    palette="Reds",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Distribution of Credit Currency",
    fontsize=14,
    fontweight="bold",
    pad=15
)
ax.set_xlabel("Credit Currency", fontsize=12)
ax.set_ylabel("Total Record Count (Log Scale)", fontsize=12)

ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3)

fig.tight_layout()
plt.show()

#### 🪙 What does `currency 1` Represents?

`currency 1` is simply the primary local money of the country where the loan took place—with its real name masked for data privacy. Because Home Credit operates in emerging markets, `currency 1` represents whichever local currency belonged to that specific region:

| Country | Local Currency |
| --- | --- |
| **Russia** | Russian Ruble (RUB) |
| **Vietnam** | Vietnamese Dong (VND) |
| **Kazakhstan** | Kazakhstani Tenge (KZT) |
| **Indonesia** | Indonesian Rupiah (IDR) |
| **Philippines** | Philippine Peso (PHP) |

---

#### Credit Types Distribution

In [0]:
credit_type_counts_df = (
    bureau
    .groupBy("CREDIT_TYPE")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    data=credit_type_counts_df,
    x="CREDIT_TYPE",
    y="count",
    hue="CREDIT_TYPE",
    legend=False,
    palette="Greens",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Distribution of Credit Type",
    fontsize=14,
    fontweight="bold",
    pad=15
)
ax.set_xlabel("Credit Type", fontsize=12)
ax.set_ylabel("Total Record Count (Log Scale)", fontsize=12)
ax.tick_params(axis="x", rotation=90)

ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3)

fig.tight_layout()
plt.show()

#### Detailed Breakdown of Bureau `CREDIT_TYPE` Categories

The `CREDIT_TYPE` column specifies the exact credit product or purpose of external loans reported to the Credit Bureau. Below is a detailed analysis of all 15 categories shown in the distribution plot, ordered by total record count (displayed on a logarithmic scale).

##### 1. Major Retail Credit Products

##### Consumer credit
* **What it means:** Point-of-sale financing or installment loans used to purchase consumer goods like mobile phones, electronics, home appliances, or furniture.
* **Risk Profile:** Moderate risk. Represents standard, everyday retail financing behavior. Since it forms ~72.9% of all bureau records, it serves as the baseline for applicant credit history.

##### Credit card
* **What it means:** Open-ended revolving credit lines provided by external banks or financial institutions.
* **Risk Profile:** Moderate to High risk depending on utilization. Unlike fixed installment loans, revolving lines allow continuous borrowing up to a limit, making spending habits and balance management key indicators of financial discipline.

##### 2. Asset-Backed & High-Risk Retail Loans

##### Car loan
* **What it means:** Collateralized auto financing used specifically to purchase personal or commercial vehicles.
* **Risk Profile:** Low to Moderate risk. Because the loan is secured by a physical asset (the vehicle), borrowers are generally motivated to maintain payments to avoid repossession.

##### Mortgage
* **What it means:** Long-term secured loans taken out to purchase residential real estate or housing.
* **Risk Profile:** Low risk. Mortgages require strict income verification and credit underwriting. A borrower maintaining a healthy mortgage history demonstrates long-term financial stability and high creditworthiness.

##### Microloan
* **What it means:** Short-term, high-interest loans typically issued by Microfinance Institutions (MFIs) or payday lenders for emergency cash needs.
* **Risk Profile:** High risk. Resorting to microloans often signals acute short-term liquidity distress or an inability to secure credit from traditional commercial banks.

##### 3. Business & Commercial Financing

##### Loan for business development
* **What it means:** Business capital extended to self-employed individuals, sole proprietors, or small business owners to expand operations or fund growth projects.
* **Risk Profile:** Variable risk. Performance depends heavily on the success of the borrower's business venture.

##### Loan for working capital replenishment
* **What it means:** Short-term commercial financing used to cover operational expenses like payroll, inventory purchases, or day-to-day cash flow shortages.
* **Risk Profile:** Moderate to High risk. High frequency of working capital borrowing can indicate recurring operating cash flow issues.

##### Loan for the purchase of equipment
* **What it means:** Asset-specific business financing used to buy machinery, tools, or vehicles required for business operations.
* **Risk Profile:** Moderate risk. Similar to auto loans, this debt is backed by the productive equipment being financed.

##### 4. Specialized, Niche, & Rare Credit Types

##### Another type of loan
* **What it means:** Valid credit products reported by financial institutions that do not fall under standard reporting categories.
* **Risk Profile:** Neutral / Ambiguous.

##### Unknown type of loan
* **What it means:** Missing, corrupted, or unclassified debt entries in the Credit Bureau system.
* **Risk Profile:** Neutral / Data Quality Flag.

##### Cash loan (non-earmarked)
* **What it means:** General-purpose personal cash loans where the borrower is not required to specify or prove the intended use of funds.
* **Risk Profile:** Moderate to High risk due to lack of asset backing or specified purpose.

##### Real estate loan
* **What it means:** Property-related financing distinct from primary residential mortgages (e.g., commercial real estate or land acquisition).
* **Risk Profile:** Low to Moderate risk (secured by property).

##### Loan for purchase of shares (margin lending)
* **What it means:** Credit extended to purchase stocks, securities, or financial investments.
* **Risk Profile:** Very High risk due to exposure to financial market volatility.

##### Mobile operator loan
* **What it means:** Micro-financing or device financing managed directly through a telecommunications provider.
* **Risk Profile:** Extremely rare (< 0.0001% of records).

##### Interbank credit
* **What it means:** Wholesale lending extended between banking institutions (typically a data entry anomaly in a retail credit dataset).
* **Risk Profile:** Outlier / Anomaly.

#### What is Credit Line?
A credit line (also called a line of credit) is a flexible financial agreement between a borrower and a lender that gives you access to a approved maximum amount of money—your credit limit.

Instead of receiving a single lump sum of cash all at once (like a standard loan), you can borrow, repay, and borrow again up to that limit whenever you need it.

---

#### CREDIT DAY OVERDUE
It measures active delinquency by tracking the exact number of days a payment is past due on a specific external loan account at the precise moment the credit report is generated. A value of `0` indicates that the loan account is fully up to date with no missed payments. In contrast, positive integers represent delinquent accounts—for instance, a value of `9` means the payment is 9 days past due, while a value of `60` signals that the borrower has missed two consecutive monthly cycles.

Unlike historical metrics that track past behavior across previous years, this field provides a real-time snapshot of ongoing financial distress. 

* **On-Time Accounts (`CREDIT_DAY_OVERDUE = 0`):** The account is healthy, and the borrower is meeting all current contractual obligations.
* **Minor Delinquencies (`1 – 30 days`):** Serves as an early warning indicator that the applicant is starting to miss recent due dates.
* **Moderate Delinquencies (`31 – 90 days`):** Demonstrates clear financial strain and an inability to maintain regular payment schedules.
* **Severe Defaults (`90+ days`):** Indicates critical financial distress, where the external loan is on the verge of charge-off, legal action, or debt sale.

In [0]:
# quantiles
percentiles = bureau.approxQuantile(
    "CREDIT_DAY_OVERDUE",
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99],
    0.01
)

percentiles

In [0]:
bureau.groupBy("CREDIT_DAY_OVERDUE") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10, truncate=False)

*Note: Using approxQuantile for performance*

---

#### Monetary Credit Amounts

Understanding these four fields depends on the product category. In our bureau dataset, the top 3 loan types—**Consumer credit**, **Credit card**, and **Car loan**—account for the vast majority of records:

##### Scenario A: Revolving Credit Line
Imagine checking your mobile banking app for an active credit card:

* **`AMT_CREDIT_SUM` = 250,000**
  * **Your Perspective:** Your **Total Credit Limit**. The maximum borrowing cap approved by the bank.
* **`AMT_CREDIT_SUM_DEBT` = 80,000**
  * **Your Perspective:** Your **Current Balance**. The total amount spent and currently owed.
* **`AMT_CREDIT_SUM_LIMIT` = 170,000**
  * **Your Perspective:** Your **Available Credit**. How much room you still have left to spend ($250,000 limit minus $80,000 balance).
* **`AMT_CREDIT_SUM_OVERDUE` = 5,000**
  * **Your Perspective:** Your **Late Bill**. Out of your $80,000 balance, a required $5,000 monthly payment was missed and is past due.

##### Scenario B: Fixed Installment Loan
Imagine taking out a fixed-term loan (like a **Car loan** or Point-of-Sale **Consumer credit** for electronics/furniture):

* **`AMT_CREDIT_SUM` = 20,000**
  * **Your Perspective:** The **Original Loan Amount** approved for the purchase.
* **`AMT_CREDIT_SUM_DEBT` = 12,000**
  * **Your Perspective:** The **Remaining Balance** left to pay off.
* **`AMT_CREDIT_SUM_LIMIT` = NULL or 0**
  * **Your Perspective:** **Not Applicable**. Fixed loans do not have an "available line to spend"—once funds are disbursed, you cannot re-borrow from them.
* **`AMT_CREDIT_SUM_OVERDUE` = 400**
  * **Your Perspective:** The **Past Due Amount**. You missed last month's $400 installment payment.

##### Key Data Insight for Missing Values
In the `bureau` dataset, `AMT_CREDIT_SUM_LIMIT` contains a high percentage of `NULL` values. This is **expected domain behavior**, as fixed installment products (like Consumer credit and Car loans, which form over 74% of the dataset) naturally do not carry a revolving credit limit.

In [0]:
bureau.select("AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE") \
      .summary("min", "1%", "5%", "25%", "50%", "75%", "90%", "95%", "99%", "max") \
      .show()

In [0]:
bureau.filter(sf.col("AMT_CREDIT_SUM_DEBT") < 0.0).select("AMT_CREDIT_SUM_DEBT").count()

In [0]:
bureau.groupBy("CREDIT_TYPE").count().show(truncate=-1)

#### Real-Data Breakdown: What the Summary Stats Tell Us

##### 1. Negative Numbers in Debt and Credit Limit
* **What it means:** A negative debt balance usually means an **overpayment** (for example, paying off a credit card bill twice by mistake or getting a large refund on a return). 
* **Data Anomaly:** Notice that the lowest negative debt (`-4,705,600.32`) matches the highest credit limit (`4,705,600.32`). This exact match means an external bank made a bookkeeping or reporting error where two columns were accidentally swapped.

##### 2. Huge Wave of Zeros (Zero-Inflation)
* **Debt (`AMT_CREDIT_SUM_DEBT` is 0 up to 70% of the time):** Over 70% of the loan records show $0 balance. This happens because the bureau tracks a person's entire loan history, and most old loans are already fully paid off.
* **Credit Limit (`AMT_CREDIT_SUM_LIMIT` is 0 up to 94% of the time):** Almost 95% of records have no available credit limit. Fixed-term loans like car loans, consumer electronics loans, and mortgages don't have a spending cap—only credit cards do.
* **Overdue Amounts (`AMT_CREDIT_SUM_OVERDUE` is 0 for over 99% of records):** Less than 1% of all reported loans currently have a late balance. Most people pay on time, making active delinquency a rare event.

##### 3. Extremely High Maximum Values
The maximum original loan size reaches ~585 million, and the maximum current debt reaches ~170 million. The dataset includes high-net-worth commercial borrowers, real estate developers, and corporate business accounts alongside regular individual applicants.

---

#### What `DAYS_CREDIT_UPDATE` Actually Tells You

`DAYS_CREDIT_UPDATE` tells you **how fresh or old the credit bureau's information is for a specific loan**. 

When external banks report loan updates to the bureau, they send periodic status reports. This field records how many days passed between that last status report and the day the client applied for the current Home Credit loan (represented as a negative number).

* **Recent updates (`0` to `-30`):** The data was reported recently (within the last month), giving you a real-time snapshot of the client's debt and overdue status.
* **Old updates (`-1000` or lower):** The data was reported years ago, meaning the record is stale historical information from an account that has likely been closed or paid off since.

In [0]:
bureau.select("DAYS_CREDIT_UPDATE") \
      .summary("min", "1%", "5%", "25%", "50%", "75%", "max") \
      .show()

#### Data Summary Breakdown for DAYS_CREDIT_UPDATE

* **Recent Activity (Median = -395 days, 75th percentile = -33 days):** Half of all loan accounts in the dataset were updated within the last ~13 months, and 25% were updated within the last month (-33 days). This means a large portion of the records reflect very recent financial activity.
* **Future-Date Data Anomaly (Max = +372 days):** Positive values indicate reports dated *after* the Home Credit loan application. A value of +372 suggests a reporting error, a placeholder default value, or a corrupted record from the bureau.
* **Extreme Outlier / Historical Record (Min = -41,947 days):** A value of -41,947 corresponds to nearly 115 years in the past. This is almost certainly an erroneous dummy/sentinel value used by an external bank to denote missing or unknown update dates.

In [0]:
bureau.orderBy("DAYS_CREDIT_UPDATE").select("DAYS_CREDIT_UPDATE").show(100)

#### Domain Insight: Origin of Extreme Negative Values (-40,000 Range)

Values falling into the **-40,000 to -42,000 range** (such as `-41,851` and `-41,947`) are **technical artifacts caused by legacy date-parsing systems**, not genuine historical credit activity (~115 years ago).
In legacy financial architectures and Microsoft Excel, day `0` is benchmarked to December 31, 1899. When systems fail to parse an uninitialized or blank date string relative to a modern application date, the delta evaluates to roughly `-41,000` days.

In [0]:
# outliers in the left tail
bureau.filter(sf.col("DAYS_CREDIT_UPDATE") <= -41851).count()

In [0]:
# positive value counts (outliers in right tail)
bureau.filter(sf.col("DAYS_CREDIT_UPDATE") > 0).count()

In [0]:
# let's see all the positive values
bureau.filter(sf.col("DAYS_CREDIT_UPDATE") > 0).select("DAYS_CREDIT_UPDATE").show()

In [0]:
bureau.filter(
    (sf.col("DAYS_CREDIT_UPDATE") > -41851) & 
    (sf.col("DAYS_CREDIT_UPDATE") <= 0)
).select("DAYS_CREDIT_UPDATE").plot.hist(title="Histogram Plot: Days Credit Update (After Outlier Removal)")

---

#### What does `AMT_ANNUITY` Tells Us?

`AMT_ANNUITY` represents the **required periodic payment amount** (typically monthly) that the borrower must pay toward a specific loan account. It measures the borrower's recurring financial obligation.

`Interpretation of Values`

* **Normal Positive Values (e.g., `5,000` or `12,000`):** The client is contractually required to pay this exact amount every month for that specific loan. Higher values indicate a heavier monthly debt burden on the client's cash flow.
* **`0.0` or Close to `0.0`:** The loan requires no active monthly installments. This occurs on fully paid-off accounts, zero-interest promotional balances, or line-of-credit accounts where no balance is drawn.
* **`NULL` / Missing Values:** Extremely common in bureau records (often exceeding 70% missingness). Missing values usually mean the external bank did not report the payment schedule to the bureau, or the product type (like a standard credit card line) has a flexible minimum payment rather than a fixed annuity.

In [0]:
# summary stats
bureau.select("AMT_ANNUITY").describe().show()

In [0]:
bureau.groupBy("AMT_ANNUITY").count().orderBy("count", ascending=False).limit(5).show()

---

#### What `CNT_CREDIT_PROLONG` Actually Tells Us?

`CNT_CREDIT_PROLONG` records the **number of times a borrower extended or postponed the deadline** on a specific credit bureau loan.

When a borrower realizes they cannot make an upcoming payment deadline, they may contact the lender to request an extension (prolongation). While this avoids an immediate formal default on their credit record, frequently asking to roll over or delay payments indicates severe cash-flow distress.

##### Key Observations & Interpretation

* **`0`:** The vast majority of loans have never been prolonged. The borrower either paid on time or went straight into delinquency without an agreed extension.
* **Low Positive Values (`1` or `2`):** The borrower asked for an occasional extension on that loan (e.g., during a temporary financial hardship).
* **High Positive Values (`3+`):** A strong warning sign of chronic financial difficulty, where the borrower relies on repeatedly extending existing debt rather than clearing it.

In [0]:
cnt_credit_prolong_df = (
    bureau
    .groupBy("CNT_CREDIT_PROLONG")
    .count()
    .orderBy("CNT_CREDIT_PROLONG")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=cnt_credit_prolong_df,
    x="CNT_CREDIT_PROLONG",
    y="count",
    hue="CNT_CREDIT_PROLONG",
    legend=False,
    palette="plasma",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Distribution of CNT_CREDIT_PROLONG",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("CNT_CREDIT_PROLONG", fontsize=12)
ax.set_ylabel("Total Record Count (Log Scale)", fontsize=12)

ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3)

fig.tight_layout()
plt.show()

---

#### DAYS_CREDIT_ENDDATE & DAYS_ENDDATE_FACT

These two features work together as a **pair** to give you the timeline of a loan: when it was *supposed* to end versus when it *actually* ended.

##### What They Actually Tell You

1. **`DAYS_CREDIT_ENDDATE` (Planned Completion Date)**
   * **Definition:** The expected remaining duration or maturity date of the loan at the time of application.
   * **Format:** Negative numbers mean the loan was **scheduled to end in the past**. Positive numbers mean the loan is **scheduled to mature in the future**.
   * **What it tells you:** How long the contractual term was, and how much longer an active loan is supposed to run.

2. **`DAYS_ENDDATE_FACT` (Actual Completion Date)**
   * **Definition:** The exact day the loan was officially closed/settled.
   * **Format:** Represented as a negative number relative to the application date.
   * **What it tells you:** When the client paid off and closed the account.
   * **Crucial Rule:** If an account is **still active**, this field will be `NULL`.

##### Understanding the Relationship

Imagine you took out a 2-year personal loan on Day `-730` (2 years ago):

* **Scenario 1: Paid off on schedule**
  * `DAYS_CREDIT_ENDDATE` = `-10` (was supposed to end 10 days ago).
  * `DAYS_ENDDATE_FACT` = `-10` (actually closed 10 days ago).
  * **Status:** Clean closed loan.

* **Scenario 2: Paid off early**
  * `DAYS_CREDIT_ENDDATE` = `+180` (was scheduled to mature 6 months in the future).
  * `DAYS_ENDDATE_FACT` = `-30` (closed 30 days ago).
  * **Status:** Early closure—shows high liquidity or refinancing.

* **Scenario 3: Still Active**
  * `DAYS_CREDIT_ENDDATE` = `+365` (1 year left on the contract).
  * `DAYS_ENDDATE_FACT` = `NULL` (account is not closed yet).
  * **Status:** Active loan currently in progress.

* **Scenario 4: Past Maturity & Unclosed (Red Flag)**
  * `DAYS_CREDIT_ENDDATE` = `-180` (was supposed to be fully paid 6 months ago).
  * `DAYS_ENDDATE_FACT` = `NULL` (still not closed).
  * **Status:** Potential default/write-off or overdue agreement.

*Note: Loan maturity is the agreed-upon date when a loan must be paid back in full, bringing the credit contract to its official end.*

##### Percentage of Loan Case Paid off on schedule.

In [0]:
bureau.filter(sf.col("DAYS_CREDIT_ENDDATE") == sf.col("DAYS_ENDDATE_FACT")).count()/bureau.count()

Let's explore it further by defining logical cases.

In [0]:
# categorize closure types and aggregate counts
closure_df = bureau.withColumn(
    "CLOSURE_TYPE", 
    sf.when(sf.col("DAYS_ENDDATE_FACT").isNull(), "Still Active")
      .when(sf.col("DAYS_ENDDATE_FACT") == sf.col("DAYS_CREDIT_ENDDATE"), "Ended On Time")
      .when(sf.col("DAYS_ENDDATE_FACT") < sf.col("DAYS_CREDIT_ENDDATE"), "Closed Early")
      .when(sf.col("DAYS_ENDDATE_FACT") > sf.col("DAYS_CREDIT_ENDDATE"), "Closed Late")
      .otherwise("Unknown")
).groupBy("CLOSURE_TYPE").count().toPandas()

# calculate %ages
total_count = closure_df["count"].sum()
closure_df["percentage"] = (closure_df["count"] / total_count) * 100
closure_df = closure_df.sort_values(by="count", ascending=False)

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=closure_df, 
    x="percentage", 
    y="CLOSURE_TYPE", 
    palette="bright",
    hue="CLOSURE_TYPE",
    edgecolor="black"
)

for p in ax.patches:
    width = p.get_width()
    ax.annotate(
        f"{width:.1f}%", 
        (width + 0.8, p.get_y() + p.get_height() / 2.),
        ha="left", va="center", fontsize=10, fontweight="bold"
    )

plt.title("Distribution of Bureau Loan Closure Types", fontsize=14, pad=15)
plt.xlabel("Percentage of Total Loans (%)", fontsize=12)
plt.ylabel("Closure Category", fontsize=12)
plt.xlim(0, max(closure_df["percentage"]) + 10)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

#### Interpretation of the Above Bar Plot

1. `Active Debt Burden (36.9% Still Active)`: Out of all the loans in the bureau records, over a third are still open right now. This is crucial because these active accounts are where the applicant’s money is currently going. They represent their current monthly payments and existing financial commitments that Home Credit needs to account for today.
2. `Proactive Repayers (28.4% Closed Early)`: More than a quarter of all loans were paid off ahead of schedule. When borrowers consistently pay off debts early, it usually means they have good cash flow, manage their money well, or had enough extra funds to clear their balance early. In credit scoring, this group generally represents a much lower default risk.
3. `Predictable Payers (18.9% Ended On Time)`: Nearly 1 in 5 loans ran its exact planned lifespan and was settled right on the target end date. These are standard, reliable borrowers who follow their payment schedule contractually without any early payoffs or late delays.
4. `Struggling or Late Payers (13.9% Closed Late)`: About 14% of loans were eventually paid off, but took longer than originally agreed. While these borrowers didn't completely default (since the loans are closed now), the delay points to past financial stress, missed deadlines, or negotiated extra time. This makes them noticeably riskier than people who paid early or on time.
5. `Missing Records (1.9% Unknown)`: A tiny fraction of accounts—less than 2%—are missing key dates, making it impossible to tell when or how they ended. These are just small reporting gaps from the reporting banks that can easily be handled during data cleaning.

In [0]:
bureau.show(5)

#### 👽 Null Values Analysis

In [0]:
# null values
null_counts = bureau.select([
    sf.count(sf.when(sf.col(c).isNull(), c)).alias(c)
    for c in bureau.columns
])
null_counts.show(truncate=False)

In [0]:
total_rows = bureau.count()

null_pct = (
    null_counts
    .toPandas()
    .T
    .reset_index()
)

null_pct.columns = ["column", "null_count"]

null_pct["null_pct"] = (
    null_pct["null_count"] / total_rows * 100
)

null_pct = null_pct[
    null_pct["null_count"] > 0
].sort_values("null_pct", ascending=False)

null_pct

In [0]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=null_pct,
    x="null_pct",
    y="column",
    hue="column",
    legend=False,
    palette="Reds",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Missing Values by Column",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Missing Values (%)")
ax.set_ylabel("Column")

ax.grid(axis="x", linestyle="--", alpha=0.7)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()

#### 📝 Missingness Report: Bureau Data Field Analysis

##### 1. `AMT_ANNUITY` (~71.5% Missing)
* **Missingness Type:** **MAR (Missing at Random)** / **MNAR (Missing Not at Random)**
* **Why it happens:** External banks simply aren't forced to share this data. Credit bureaus like Equifax or TransUnion strictly enforce tracking total loan amounts and late payments, but scheduled monthly payments are often treated as optional metadata. When banks export bulk data, they frequently skip optional fields.
* **Product Reality:** Revolving credit cards don't have a fixed monthly annuity to begin with—your payment changes based on what you spend. Similarly, bullet or deferred-payment loans expect a single lump sum at the end, so a regular monthly payment number doesn't even exist for them.

##### 2. `AMT_CREDIT_MAX_OVERDUE` (~65.5% Missing)
* **Missingness Type:** **MNAR (Missing Not at Random)**
* **Why it happens:** "No news is good news." For responsible borrowers who pay on time every single month, a "maximum overdue amount" was simply never triggered. Instead of writing `$0.00` across millions of clean accounts, reporting systems leave the field blank. In credit scoring, a blank here strongly points to a clean payment history.

##### 3. `DAYS_ENDDATE_FACT` (~36.9% Missing)
* **Missingness Type:** **MNAR (Missing Not at Random)**
* **Why it happens:** The loan is still open. If someone is currently paying off a mortgage, personal loan, or using an active credit card, the account obviously hasn't closed yet. The fact that this field is blank isn't a data error—it is a critical business signal telling us the applicant has an ongoing financial commitment right now.

##### 4. `AMT_CREDIT_SUM_LIMIT` (~34.5% Missing)
* **Missingness Type:** **MNAR (Missing Not at Random)**
* **Why it happens:** It comes down to how the loan is built. Fixed installment loans (like a 3-year auto loan or personal loan) give you a lump sum upfront—they don't have a revolving credit limit. Because a flexible spending limit doesn't exist for standard installment products, banks report it as `NULL` rather than zero. You'll generally only see non-null values here for revolving credit cards or open lines of credit.

##### 5. `AMT_CREDIT_SUM_DEBT` (~15.0% Missing)
* **Missingness Type:** **MAR (Missing at Random)**
* **Why it happens:** When a loan is fully paid off and closed, the debt balance drops to `$0.00`. However, once an account status changes to "Closed," many banks stop sending active balance updates altogether. Instead of logging zero every month for old historical accounts, legacy data pipelines export them as blank values.

#### Duplication Check

In [0]:
# check for duplicate instances
total_rows = bureau.count()
distinct_rows = bureau.distinct().count()

duplicate_rows = total_rows - distinct_rows

print(f"Duplicate rows: {duplicate_rows:,}")

### 📊 Bureau Balance
The `bureau_balance` table is a monthly statement history for the external loans listed in the bureau table. Each row represents a single month's snapshot for one specific external loan, tracking whether the customer paid on time, how many days their payment was overdue, or if the account was closed during that month.

In [0]:
bureau_bal = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_DIR}/bureau_balance.csv")
)

In [0]:
# shape
print(f"({bureau_bal.count()}, {len(bureau_bal.columns)})")

In [0]:
# schema
bureau_bal.printSchema()

#### ✏️ Table Description

##### 1. SK_ID_BUREAU
* **Description:** Unique identifier for each credit record reported by the Credit Bureau.
* **Interpretation:** This serves as our foreign key to link monthly payment histories back to specific loans listed in the `bureau` table.

##### 2. MONTHS_BALANCE
* **Description:** Month of the balance snapshot relative to the current application date.
* **Interpretation:** Time location represented as negative integers. For instance, `0` indicates the current month's record, `-1` means last month, and `-20` means 20 months prior.

##### 3. STATUS
* **Description:** Monthly credit status of the Credit Bureau loan.
* **Interpretation:** Categorical indicator showing payment behavior for that month:
  * `0`: No delinquency (paid on time)
  * `1`: 1 to 30 days past due
  * `2`: 31 to 60 days past due
  * `3`: 61 to 90 days past due
  * `4`: 91 to 120 days past due
  * `5`: 121+ days past due or written off
  * `C`: Closed loan
  * `X`: Unknown status / no data reported for the month

In [0]:
bureau_bal.show(5)

#### Distribution of Credit Bureau Monthly Statuses

In [0]:
# aggregate on PySpark cluster & collect only summary rows
status_counts_df = bureau_bal.groupBy("STATUS").count().orderBy("count", ascending=False).toPandas()

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=status_counts_df, 
    x="STATUS", 
    y="count", 
    hue="STATUS", 
    legend=False, 
    palette="plasma", 
    ax=ax,
    edgecolor="black"
)

ax.set_title("Distribution of Credit Bureau Monthly Statuses", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Status Code", fontsize=12)
ax.set_ylabel("Total Record Count (Log Scale)", fontsize=12)
ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add text annotations directly onto the axis
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3)

# Adjust layout and render
fig.tight_layout()
plt.show()

In [0]:
# we can also use Databricks Display
# display(bureau_bal.groupBy("STATUS").count())

#### Histogram Plot of Month Balance

In [0]:
# Histogram plot
bureau_bal.select("MONTHS_BALANCE").plot.hist(title="Histogram Plot of MONTHS_BALANCE")

*Note: For the histogram plot, I did not use Pandas because doing so would require loading millions of rows into memory. Instead, the histogram was generated using a more memory-efficient approach directly on the distributed dataset.*

##### Example Record Interpretation (`bureau_balance`)

For the record with `SK_ID_BUREAU = 5715448`, `MONTHS_BALANCE = -20`, and `STATUS = 2`, the data indicates that 20 months prior to submitting their current Home Credit application, the client was 31 to 60 days late on a monthly payment for an external loan reported by the Credit Bureau. From a credit risk perspective, a status of `2` represents a moderate delinquency, signaling that the borrower experienced genuine financial distress during that particular month. However, because this occurred nearly two years ago (`MONTHS_BALANCE = -20`), it reflects historical rather than immediate behavior, meaning a predictive model should weigh it less severely than a recent delinquency—provided the client's subsequent monthly records demonstrate a recovery back to on-time payments (`STATUS = 0` or `C`).

*Note: Delinquency in banking and finance means failing to make a required debt payment on or before its official due date.*

#### How Banks get these features?
1. Every month, outside banks (like Chase, Citi, etc.) automatically send 
   their customer payment records to the Credit Bureau.
   
2. A client walks into Home Credit and applies for a new loan.
   
3. Home Credit automatically queries the Credit Bureau using the applicant's ID.
   
4. The Credit Bureau sends back this verified monthly track record (bureau_balance).

#### Understanding `MONTHS_BALANCE == 0`
In `bureau_balance`, `MONTHS_BALANCE == 0` represents the credit report recorded in the exact month the customer applied for the current loan at Home Credit. Because all historical records count backward in negative months, having a record at `0` indicates that the external credit account is actively open and reporting at the time of application, whereas accounts closed in the past terminate earlier at negative values. Financially, the `STATUS` value at `MONTHS_BALANCE == 0` serves as a real-time snapshot of the borrower's credit standing, revealing whether they are currently up to date on payments or actively delinquent on existing debt when requesting new credit.

In [0]:
bureau_bal.filter(sf.col("MONTHS_BALANCE")==0).count()

In [0]:
bureau_bal.filter(sf.col("MONTHS_BALANCE")==0) \
.groupBy("STATUS") \
.count() \
.show()

#### Status of Current Credit Report 
1. Predominantly Healthy Base (~77.5%): Combined, C (Closed) and 0 (On-Time) make up 473,753 accounts. The vast majority of active accounts are in good standing right when clients apply.

2. Active Delinquencies (~1.02% / 6,258 Accounts): These rows represent accounts where the borrower is currently behind on payments with other institutions (STATUS 1 through 5) at the exact moment they are applying for a loan with Home Credit. Notice that 5 (976 accounts) is higher than 2, 3, and 4 combined—these are persistent non-performing loans or write-offs.

3. High Proportion of Unreported Data (X = ~21.4%): Over 130k accounts have no status recorded in month 0, which commonly occurs with revolving credit facilities, inactive accounts, or delayed bureau reporting.

##### Null Values

In [0]:
# null values
bureau_bal.select([
    sf.count(sf.when(sf.col(c).isNull(), c)).alias(c)
    for c in bureau_bal.columns
]).show(truncate=False)

##### Duplicate Values

In [0]:
# check for duplicate instances
total_rows = bureau_bal.count()
distinct_rows = bureau_bal.distinct().count()

duplicate_rows = total_rows - distinct_rows

print(f"Duplicate rows: {duplicate_rows:,}")